<a href="https://colab.research.google.com/github/ArturStachnik/WorkshopPaleoML/blob/main/sesion_4/H4_programar_con_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Programar con IA

**Machine learning y paleoclima · Red Paleonet · Granada, 2026**

Aquí el código lo escribe la IA. El trabajo es pedir bien, comprobar lo que devuelve y decidir. El cuaderno no trae soluciones: trae los pasos de cualquier proyecto con datos y, en cada uno, qué pedir y qué comprobar.

| | Paso | |
|---|---|---|
| 1 | **Conocer los datos** | Análisis exploratorio, sin tocar nada |
| 2 | **Preparar los datos** | Las decisiones que no se delegan |
| 3 | **Plantear el problema** | Qué se predice, con qué, y una línea base |
| 4 | **Validar** | Separar bien lo que el modelo ve de lo que no |
| 5 | **Entender el modelo** | Si sobra tiempo |
| 6 | **Dejarlo reproducible** | Si sobra tiempo |

**Datos.** Los tuyos, en CSV o Excel. Si no traes, **99.774 muestras de polen de 1.446 sondeos europeos**, casi todos de lagos y turberas (LegacyPollen 2.0), con una pregunta: **¿se puede datar una muestra por su polen?**

**IA.** Cualquiera: Gemini, que está dentro de Colab, Claude, ChatGPT… El código se pega en las celdas *Pega aquí*. Antes de subir datos propios a un asistente externo, repasa la diapositiva de buenas prácticas.

Debajo de cada paso, plegado, lo que salió al probarlo con el polen. Ábrelo después de intentarlo.

---
## 0. Preparar

1. **Archivo → Guardar una copia en Drive** (*File → Save a copy in Drive*).
2. Elegir los datos en la celda de abajo y ejecutarla. Con *Mis propios datos* aparece un botón para subir el fichero.

La tabla queda en una variable llamada `datos`. Ese es el nombre que hay que darle a la IA.

In [ ]:
#@title Cargar los datos
DATOS = "Polen de Europa (LegacyPollen 2.0)"  #@param ["Polen de Europa (LegacyPollen 2.0)", "Mis propios datos (CSV o Excel)"]
import io, os
import pandas as pd
pd.set_option("display.max_columns", None)
REPO = "https://raw.githubusercontent.com/ArturStachnik/WorkshopPaleoML/main/datos/"


def ruta(fichero):
    for base in ("datos/", "../datos/"):
        if os.path.exists(base + fichero):
            return base + fichero
    return REPO + fichero


if DATOS.startswith("Polen"):
    datos = pd.read_csv(ruta("polen_europa.csv.gz"))
    taxones = pd.read_csv(ruta("polen_europa_taxones.csv"))
else:
    from google.colab import files
    nombre, contenido = next(iter(files.upload().items()))
    if nombre.lower().endswith((".xlsx", ".xls")):
        datos = pd.read_excel(io.BytesIO(contenido))
    else:
        try:
            texto = contenido.decode("utf-8")
        except UnicodeDecodeError:
            texto = contenido.decode("latin-1")
        datos = pd.read_csv(io.StringIO(texto), sep=None, engine="python")

print(f"{len(datos):,} filas y {datos.shape[1]} columnas".replace(",", "."))
datos.head()

### La ficha de los datos

Es lo primero que se pega en la conversación: el **contexto** y los **datos** del prompt. Las siete partes completas están en la chuleta del final.

Con el polen:

```
Trabajo con polen fósil de sondeos europeos (LegacyPollen 2.0, Li et al. 2025, PANGAEA).
Uso Python en Google Colab y la tabla está cargada en un DataFrame de pandas llamado `datos`.

Cada fila es una muestra de un sondeo; cada sondeo tiene varias. Columnas:
- id_sitio, sitio, latitud, longitud, altitud_m: el sondeo.
- archivo: tipo de depósito (Lake, Peat…). Puede faltar.
- profundidad_m: profundidad de la muestra, en metros.
- edad_ka: edad en miles de años antes de 1950 (mediana del modelo edad-profundidad);
  edad_min_ka y edad_max_ka son sus límites. Puede faltar.
- De Pinus a Olea: porcentaje de polen de los 30 taxones principales; otros, el resto.
- Abreviaturas: Poac = Poaceae, Cyperac = Cyperaceae, Asterac = Asteraceae, Jun = Juniperus,
  Frx = Fraxinus, Rosae = Rosaceae, Ranae = Ranunculaceae, Brassicac = Brassicaceae,
  Cphae = Caryophyllaceae.
```

Con datos propios:

```
Trabajo con [qué registro, de dónde, qué periodo] para [qué quiero averiguar].
Uso Python en Google Colab y la tabla está cargada en un DataFrame de pandas llamado `datos`.
Cada fila es [una muestra, un año…]. Columnas:
- [columna]: [qué es], en [unidades]. [Cómo están marcados los huecos].
```

**Para probar:** pregúntale qué es la columna `Frx` sin la ficha y luego con ella. Lo que adivine sin contexto es lo que el código dará por supuesto sin avisar.

---
# 1. Conocer los datos

**Qué se busca.** Las cinco preguntas de la sesión 2: forma, aspecto, rangos, huecos y distribución. Aquí no se cambia nada.

> Quiero un análisis exploratorio de `datos` antes de tocar nada. Dime cuántas filas y sondeos hay, qué columnas tienen huecos y cuántos, y el rango de cada variable numérica. Dibuja un mapa de los sondeos con el número de muestras de cada uno y un histograma de `edad_ka`. No borres ni cambies ninguna fila; si ves algo raro, haz una lista.

**Qué comprobar.**
* Que las cifras cuadran con lo que sabes: zona, edades, unidades.
* Que no ha limpiado por su cuenta. `dropna()` borra cualquier fila con un hueco, aunque sea en una columna que no se va a usar.

In [ ]:
# Pega aquí el código de la IA. Para más celdas, + Código (+ Code).

<details>
<summary><b>Lo que salió con el polen</b></summary>

* 99.774 muestras de 1.446 sondeos; mediana de 53 muestras por sondeo.
* Sin edad, el 11 %. Sin tipo de archivo, el 15 %.
* `dropna()` se lleva el 25 % de la tabla, la mayoría por la columna `archivo`, que no hace falta.
* 1.162 edades negativas: son posteriores a 1950, no un error.
* Edades de hasta 272 ka en 17 sondeos largos; casi todo está entre 0 y 15 ka.
* 1.987 muestras repiten la profundidad de otra del mismo sondeo. Mirar antes de llamarlas duplicados.

</details>

---
# 2. Preparar los datos

**Qué se busca.** Una tabla lista para el modelo. Las decisiones son tuyas: qué periodo, qué hacer con cada hueco y cada valor raro, qué transformación y qué entra como predictor.

> Prepara una tabla para el modelo a partir de `datos`. Quédate con las muestras con edad entre 0 y 15 ka; las que no tienen edad se quedan fuera, no las rellenes. No uses la columna `archivo` ni quites filas por sus huecos. Aplica la raíz cuadrada a los porcentajes de polen. Dime cuántas muestras y sondeos quedan y cuántas han salido por cada motivo.

**Qué comprobar.** El recuento de cada filtro. Y que no ha rellenado la variable que se quiere predecir: inventar la respuesta es lo único que no se puede hacer.

In [ ]:
# Pega aquí el código de la IA. Para más celdas, + Código (+ Code).

<details>
<summary><b>Lo que salió con el polen</b></summary>

Quedan **85.277 muestras de 1.319 sondeos**.

</details>

---
# 3. Plantear el problema

**Qué se busca.** La pregunta en una frase (*estimar `edad_ka` a partir del polen, para datar muestras sin fechas*) y empezar por lo simple: predecir siempre la media, luego una regresión lineal, luego un modelo flexible. Cada uno tiene que ganar al anterior.

> Quiero estimar `edad_ka` a partir de los porcentajes de polen. Dame el error (RMSE, en ka) de predecir siempre la media, de una regresión lineal y de un `HistGradientBoostingRegressor`, en una tabla. Fija la semilla aleatoria.

**Qué comprobar.** Que el error está en unidades que entiendes y se puede comparar con algo conocido, como la incertidumbre del modelo de edad. Un random forest con estos datos tarda varios minutos en Colab.

In [ ]:
# Pega aquí el código de la IA. Para más celdas, + Código (+ Code).

<details>
<summary><b>Lo que salió con el polen</b></summary>

Media 3,9 ka · regresión lineal 2,6 ka · boosting 1,9 ka, con sondeos que el modelo no ha visto. La distancia entre la edad mínima y la máxima del modelo de edad es de 0,65 ka en la muestra típica.

</details>

---
# 4. Validar

**Qué se busca.** El error con datos que el modelo no ha visto. Las muestras de un mismo sondeo se parecen; si se separan al azar, el modelo reconoce el sondeo en vez de la época y acierta de más. Hay que **separar por sondeo** (`GroupKFold`). Con datos propios, lo mismo con la cueva, la estación o el año.

> Valida el modelo con validación cruzada de 5 bloques separando por sondeo (`GroupKFold`, con `id_sitio` como grupo). Compara el error con el de una separación aleatoria por muestras y explícame la diferencia.

**Qué comprobar.**
* Pregunta siempre **cómo ha separado los datos**. `train_test_split` sin más es al azar.
* Que nada (escalar, elegir variables, ajustar hiperparámetros) se ha hecho con todos los datos antes de separar.
* Desconfía de un error demasiado bueno.

In [ ]:
# Pega aquí el código de la IA. Para más celdas, + Código (+ Code).

<details>
<summary><b>Lo que salió con el polen</b></summary>

| Modelo | Al azar | Por sondeo |
|---|---|---|
| Boosting, solo polen | 1,55 ka | 1,85 ka |
| k-NN, polen y coordenadas | **1,14 ka** | **1,97 ka** |

Al azar, el k-NN encuentra la muestra vecina del mismo sondeo y parece el mejor; con sondeos nuevos es el peor. **Si te sale cerca de 1 ka, revisa cómo se han separado los datos.**

</details>

---
# 5. Entender el modelo (si sobra tiempo)

**Qué se busca.** Qué variables usa, si cuadra con lo que se sabe y dónde falla, como en la sesión 3.

> Calcula la importancia por permutación de cada taxón con los datos de prueba. Dibuja el error frente a la edad real y un mapa con el error medio de cada sondeo. No interpretes la ecología, eso lo hago yo.

**Qué comprobar.** Que la lectura ecológica es tuya. La IA escribe de ecología con soltura y se equivoca con la misma soltura.

In [ ]:
# Pega aquí el código de la IA. Para más celdas, + Código (+ Code).

<details>
<summary><b>Lo que salió con el polen</b></summary>

* Pesan *Fagus*, *Alnus*, *Artemisia*, *Ulmus* y *Quercus*: *Artemisia* y *Pinus* en el Tardiglacial; *Corylus*, *Ulmus* y *Tilia* en el Holoceno temprano y medio; *Fagus* y *Carpinus* en el reciente.
* El modelo envejece lo reciente (+1,2 ka en los últimos 2 ka) y rejuvenece lo antiguo (−1,7 ka entre 12 y 15 ka): regresión hacia la media.
* **Para pensar:** las coordenadas ayudan un poco incluso separando por sondeo. ¿Por qué? La vegetación no cambió a la vez en toda Europa.

</details>

---
# 6. Dejarlo reproducible (si sobra tiempo)

Semilla fijada, **Entorno de ejecución → Reiniciar sesión y ejecutar todo** (*Runtime → Restart session and run all*) sin errores, versiones de las librerías, los prompts importantes guardados en celdas de texto y el uso de la IA declarado al publicar.

> Revisa el cuaderno para que se ejecute de arriba abajo en una sesión nueva, con la semilla fijada en todo lo que tenga azar. Añade al final una celda que imprima las versiones de Python y de las librerías usadas. No cambies nada más.

In [ ]:
# Pega aquí el código de la IA. Para más celdas, + Código (+ Code).

---
# Chuleta para llevarse

**Las siete partes de un buen prompt**

| Parte | Responde a |
|---|---|
| **Contexto** | ¿Quién soy, con qué trabajo y para qué? |
| **Datos** | ¿Qué hay en la tabla y de dónde sale? |
| **Objetivo** | ¿Qué quiero exactamente? |
| **Criterio científico** | ¿Qué sé yo que el modelo no sabe? |
| **Restricciones** | ¿Qué tiene que respetar? |
| **Comprobaciones** | ¿Cómo sé que está bien? |
| **Formato** | ¿Cómo quiero la respuesta? |

**Cuatro hábitos:** leer antes de ejecutar, entender el error, comprobar con lo que sé y controlar los cambios.

| Paso | Lo que hay que vigilar |
|---|---|
| Conocer los datos | Que no limpie nada por su cuenta |
| Preparar | Filas que desaparecen en silencio |
| Plantear | Empezar por lo complejo |
| Validar | Muestras del mismo grupo a los dos lados |
| Entender | Dar por buena su interpretación |
| Reproducir | Semilla, orden de las celdas, versiones |

---

*Datos: Li, C. et al. (2025). LegacyPollen2.0: Taxonomically harmonized pollen percentages of European samples with revised chronologies. PANGAEA, https://doi.org/10.1594/PANGAEA.971693 (CC BY 4.0). Tipo de archivo de https://doi.org/10.1594/PANGAEA.971802. La tabla del taller conserva los 30 taxones principales y suma el resto en `otros`.*